<div style="display:flex; align-items:center; gap:10px; margin-bottom:8px;">
  <span style="font-size:26px; color:#9558B2;">●</span>
  <span style="font-size:26px; color:#389826;">●</span>
  <span style="font-size:26px; color:#CB3C33;">●</span>
  <span style="font-size:26px; color:#4063D8;">●</span>
  <span style="font-size:30px; font-weight:700; margin-left:6px;">Julia</span>
</div>

# Julia с нуля — **Lesson 12**
## 📘 **Factorizations and Other Fun** — факторизации, структуры матриц и обобщённая линейная алгебра

**Cartesian School · Julia Course**  
**Автор:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School


## Информация об уроке

| Поле | Значение |
|---|---|
| Курс | Julia с нуля |
| Номер урока | Lesson 12 |
| Название | Factorizations and Other Fun |
| Уровень | Средний |
| Ориентировочное время | 300–420 минут |
| Требования | Lesson 0–11 |
| Темы | LU, QR, Cholesky, SVD, EVD, Schur, Jordan, повторное использование факторизации, специальные матрицы, generic linear algebra, рациональные числа, точные вычисления |
| Автор | Siergej Sobolewski |
| Права | © 2026 Cartesian School |

---

## План урока

1. Зачем нужны факторизации.
2. Общее окружение `LinearAlgebra`.
3. LU.
4. Pivoting и перестановки в LU.
5. LU для систем уравнений.
6. Повторное использование факторизации.
7. Определитель из LU.
8. QR.
9. QR и least squares.
10. Cholesky.
11. Условия применимости Cholesky.
12. SVD.
13. Восстановление из SVD.
14. Низкоранговая аппроксимация.
15. EVD.
16. Eigenvalues и eigenvectors.
17. EVD и симметричные матрицы.
18. Schur decomposition.
19. Schur vs Jordan.
20. Jordan normal form — численные ограничения.
21. Символьные вычисления формы Жордана.
22. Специальные матрицы.
23. `Diagonal`.
24. `UpperTriangular` / `LowerTriangular`.
25. `Symmetric` / `Hermitian`.
26. `Bidiagonal`, `Tridiagonal`, `SymTridiagonal`.
27. Преимущества специальных структур.
28. Generic linear algebra.
29. Рациональные числа.
30. Точные линейные системы.
31. Сравнение Float64 и Rational.
32. Выбор факторизации под задачу.
33. Типичные ошибки.
34. Практика.
35. Мини-проект.
36. Checkpoint.
37. Итоги.


## Учебный стандарт Cartesian School

| Обозначение | Значение |
|---|---|
| **Цель** | чему вы научитесь в данном фрагменте |
| **Теория** | определения и правила |
| **Пример** | минимальный работающий код |
| **Анализ** | объяснение работы |
| **Важно** | правило, требующее особого внимания |
| **Типичная ошибка** | распространённая ошибка и её причина |
| **Попробуйте сами** | небольшой эксперимент |
| **Практика** | задание для самостоятельного выполнения |
| **Итоги** | ключевые выводы |


## Цели урока

После завершения Lesson 12 вы сможете:

- объяснить, почему факторизации важнее явного вычисления обратной матрицы;
- применять LU, QR, Cholesky, SVD и EVD;
- понимать роль pivoting;
- повторно использовать одну факторизацию для нескольких правых частей;
- применять Schur decomposition;
- объяснять, почему Jordan form сложна с численной точки зрения;
- использовать специальные структуры матриц;
- выполнять вычисления с типами, отличными от `Float64`;
- решать точные системы на рациональных числах;
- выбирать факторизацию в соответствии со структурой задачи.


## **1. Зачем нужны факторизации?**

### Теория

Факторизация матрицы — это представление матрицы в виде произведения более простых множителей.

Речь идёт не только о математической элегантности. Факторизации лежат в основе эффективной численной линейной алгебры.

Основные применения:

- решение систем уравнений;
- задачи наименьших квадратов;
- вычисление определителей;
- поиск собственных значений;
- анализ ранга;
- сжатие и аппроксимация данных;
- многократное решение систем с одной и той же матрицей.


### Основная идея

Вместо того чтобы многократно решать:

\[
Ax=b_1,\quad Ax=b_2,\quad Ax=b_3
\]

с нуля, можно сначала вычислить факторизацию:

\[
A = F
\]

а затем использовать один и тот же объект `F` для разных правых частей.


## **2. Окружение `LinearAlgebra`**


In [ ]:
using LinearAlgebra
using Random

Random.seed!(42)

A = rand(3, 3)
x = fill(1.0, 3)
b = A * x

@show A
@show x
@show b


В следующих разделах мы будем работать с объектами факторизаций, возвращаемыми модулем `LinearAlgebra`.


## **3. LU factorization**

### Теория

Для общей квадратной матрицы LU с учётом pivoting представляет матрицу в форме, связанной с:

\[
PA = LU
\]

где:

- `P` — матрица перестановки;
- `L` — нижнетреугольная матрица;
- `U` — верхнетреугольная матрица.


In [ ]:
A = [
    4.0 3.0
    6.0 3.0
]

F = lu(A)

@show F
@show F.L
@show F.U
@show F.p


### Важно

Не следует автоматически предполагать, что `A == L*U`.

В практической LU-факторизации часто используется pivoting, поэтому необходимо учитывать перестановку.


## **4. Pivoting и перестановки в LU**

Pivoting повышает численную устойчивость, выбирая более подходящие ведущие элементы во время исключения.


In [ ]:
A = [
    0.0 2.0
    1.0 3.0
]

F = lu(A)

@show F.p
@show F.L
@show F.U


### Анализ

Матрица с нулевым элементом в левом верхнем углу требует перестановки строк, чтобы классический алгоритм исключения мог корректно продолжить работу.


## **5. LU для решения систем**


In [ ]:
A = [
    3.0 1.0
    1.0 2.0
]

b = [9.0, 8.0]

F = lu(A)
x = F \ b

@show x
@assert A * x ≈ b


### Важно

`F \ b` использует уже вычисленную факторизацию.

Это особенно важно, когда нужно решить несколько систем с одной и той же матрицей `A`.


## **6. Повторное использование факторизации**


In [ ]:
A = rand(100, 100)
F = lu(A)

b1 = rand(100)
b2 = rand(100)
b3 = rand(100)

x1 = F \ b1
x2 = F \ b2
x3 = F \ b3

@assert A * x1 ≈ b1
@assert A * x2 ≈ b2
@assert A * x3 ≈ b3


### Анализ

Самая дорогая часть — вычисление факторизации — выполняется только один раз.

Каждое последующее решение использует уже готовые множители.


## **7. Определитель из LU**

Для квадратной матрицы определитель можно получить также на основе факторизации.


In [ ]:
A = [
    4.0 3.0
    6.0 3.0
]

F = lu(A)

@show det(A)
@show det(F)
@assert det(A) ≈ det(F)


## **8. QR factorization**

### Теория

QR представляет матрицу в виде:

\[
A = QR
\]

где:

- `Q` имеет ортонормированные столбцы;
- `R` — верхнетреугольная матрица.


In [ ]:
A = rand(5, 3)

F = qr(A)

Q = Matrix(F.Q)
R = F.R

@show size(Q)
@show size(R)
@show Q' * Q


### Важно

QR особенно важна для:

- least squares;
- ортогонализации;
- устойчивых численных методов.


## **9. QR и least squares**


In [ ]:
A = [
    1.0 1.0
    1.0 2.0
    1.0 3.0
    1.0 4.0
]

b = [1.1, 1.9, 3.2, 4.1]

F = qr(A)
x = F \ b

@show x
@show norm(A * x - b)


### Хорошая практика

Для least squares не формируйте вручную нормальные уравнения `(A' * A) \ (A' * b)` без конкретной причины.

QR обычно является более устойчивым численным подходом.


## **10. Cholesky factorization**

### Теория

Для симметричной положительно определённой матрицы:

\[
A = LL^T
\]

а в комплексном случае:

\[
A = LL^*
\]


In [ ]:
A = [
    4.0 1.0
    1.0 3.0
]

F = cholesky(Symmetric(A))

@show F.L
@show F.U


### Преимущества

Cholesky очень эффективна для SPD-матриц и требует меньше вычислений, чем общая LU-факторизация.


## **11. Условия применимости Cholesky**

Матрица должна быть положительно определённой.


In [ ]:
A_good = [
    4.0 1.0
    1.0 3.0
]

A_bad = [
    1.0 2.0
    2.0 1.0
]

@show isposdef(A_good)
@show isposdef(A_bad)


### Безопасная диагностика ошибки


In [ ]:
cholesky_failed = try
    cholesky(Symmetric(A_bad))
    false
catch
    true
end

@show cholesky_failed


## **12. SVD — Singular Value Decomposition**

### Теория

SVD раскладывает матрицу:

\[
A = U\Sigma V^*
\]

и применима также к прямоугольным матрицам.


In [ ]:
A = rand(5, 3)

F = svd(A)

@show F.S
@show size(F.U)
@show size(F.Vt)


### SVD является фундаментальным инструментом для:

- анализа ранга;
- вычисления псевдообратной матрицы;
- сжатия;
- PCA;
- низкоранговой аппроксимации;
- диагностики плохо обусловленных задач.


## **13. Восстановление из SVD**


In [ ]:
A = rand(5, 3)
F = svd(A)

A_reconstructed = F.U * Diagonal(F.S) * F.Vt

@show norm(A - A_reconstructed)
@assert A ≈ A_reconstructed


## **14. Низкоранговая аппроксимация**

Сохраняя только крупнейшие singular values, можно получить приближение исходной матрицы.


In [ ]:
A = rand(20, 10)
F = svd(A)

k = 3

A_k = F.U[:, 1:k] * Diagonal(F.S[1:k]) * F.Vt[1:k, :]

@show rank(A_k)
@show norm(A - A_k)


### Анализ

Это лежит в основе многих методов сжатия и уменьшения размерности.


## **15. EVD — eigendecomposition**

### Теория

Для квадратной матрицы ищем:

\[
Av = \lambda v
\]

где `λ` — собственное значение, а `v` — собственный вектор.


In [ ]:
A = [
    2.0 1.0
    1.0 2.0
]

E = eigen(A)

@show E.values
@show E.vectors


## **16. Проверка собственной пары**


In [ ]:
λ = E.values[1]
v = E.vectors[:, 1]

@show A * v
@show λ * v
@assert A * v ≈ λ * v


## **17. EVD и симметричные матрицы**

Для вещественной симметричной матрицы:

- собственные значения вещественны;
- собственные векторы можно выбрать ортонормированными.


In [ ]:
A = Symmetric([
    4.0 1.0
    1.0 2.0
])

E = eigen(A)

@show E.values
@show E.vectors' * E.vectors


### Хорошая практика

Если матрица действительно симметрична, передача `Symmetric(A)` позволяет использовать более подходящие алгоритмы.


## **18. Schur decomposition**

### Теория

Разложение Шура представляет матрицу в виде:

\[
A = Q T Q^*
\]

где `Q` — унитарная/ортогональная матрица, а `T` имеет треугольную или квазитреугольную форму.


In [ ]:
A = rand(4, 4)

S = schur(A)

@show S.values
@show size(S.Z)
@show size(S.T)


### Применение

Schur decomposition играет важную роль в устойчивых алгоритмах вычисления собственных значений и функций от матриц.


## **19. Schur vs Jordan**

Jordan normal form важна с теоретической точки зрения, однако разложение Шура обычно гораздо лучше подходит для численных вычислений.

Почему?

- Schur численно устойчивее;
- Jordan form чрезвычайно чувствительна к малым возмущениям;
- в вычислениях с плавающей точкой даже небольшой шум может изменить структуру жордановых блоков.


## **20. Jordan normal form — численные ограничения**

### Важно

`LinearAlgebra` не предоставляет общей численной функции `jordan(A)` как стандартного инструмента.

Это не случайное отсутствие. Jordan form сложна и неустойчива в численных вычислениях.

В практической численной работе обычно предпочитают:

- Schur;
- eigen;
- SVD.


### Когда форма Жордана имеет смысл?

- в символьной алгебре;
- в теоретическом анализе;
- в точной арифметике;
- при изучении структуры линейных операторов.


## **21. Символьная форма Жордана — концептуальный пример**

Внешние пакеты могут предоставлять символьные или точные инструменты, однако их API зависит от конкретного пакета и версии.

Поэтому в рамках курса мы не устанавливаем автоматически `JordanForm.jl`, `Symbolics.jl` или аналогичные пакеты.


In [ ]:
# Przykład koncepcyjny — wymaga zewnętrznego pakietu:
#
# using Pkg
# Pkg.add("JordanForm")
#
# Następnie należy używać API zgodnego z aktualną dokumentacją pakietu.


### Важно

Не смешивайте:

- численные вычисления `Float64`;
- символьные вычисления;
- точную рациональную арифметику.

Это разные модели вычислений с разными свойствами.


## **22. Специальные структуры матриц**

Julia умеет сохранять информацию о структуре матрицы непосредственно в её типе.


### Почему это важно?

Если известно, что матрица:

- диагональная;
- треугольная;
- симметричная;
- эрмитова;
- трёхдиагональная,

нет необходимости трактовать её как произвольную плотную `Matrix`.


## **23. `Diagonal`**


In [ ]:
D = Diagonal([1.0, 2.0, 3.0])

@show D
@show typeof(D)


In [ ]:
x = [10.0, 20.0, 30.0]

@show D * x


### Анализ

`Diagonal` хранит только элементы диагонали, а не все нули вне неё.


## **24. `UpperTriangular` и `LowerTriangular`**


In [ ]:
A = [
    1.0 2.0 3.0
    4.0 5.0 6.0
    7.0 8.0 9.0
]

U = UpperTriangular(A)
L = LowerTriangular(A)

@show U
@show L


## **25. `Symmetric` и `Hermitian`**


In [ ]:
A = [
    2.0 1.0
    1.0 3.0
]

S = Symmetric(A)

@show S
@show typeof(S)


In [ ]:
Z = [
    2 + 0im  1 + 2im
    1 - 2im  4 + 0im
]

H = Hermitian(Z)

@show H


## **26. `Bidiagonal`, `Tridiagonal`, `SymTridiagonal`**


In [ ]:
d = [2.0, 2.0, 2.0, 2.0]
e = [-1.0, -1.0, -1.0]

T = SymTridiagonal(d, e)

@show T
@show typeof(T)


### Применение

Трёхдиагональные матрицы встречаются, в частности, в:

- дискретизации дифференциальных уравнений;
- методе конечных элементов;
- цепочечных моделях;
- задачах на собственные значения.


## **27. Преимущества специальных структур**

Специальная структура может обеспечивать:

- меньшее потребление памяти;
- меньшее число операций;
- специализированный solver;
- более явный математический контракт.


### Типичная ошибка

Не преобразуйте без необходимости:

```julia
Matrix(D)
```

если алгоритм умеет работать непосредственно с `Diagonal`.


## **28. Generic linear algebra**

### Теория

Одна из сильных сторон Julia заключается в том, что многие алгоритмы линейной алгебры работают с типами, отличными от `Float64`.

Можно использовать, например:

- `Float32`;
- `BigFloat`;
- `Complex`;
- рациональные числа;
- некоторые собственные числовые типы.


## **29. Рациональные числа**

В Julia рациональные числа записываются следующим образом:


In [ ]:
a = 1 // 3
b = 2 // 5

@show a
@show b
@show a + b
@show typeof(a)


### Важно

`1//3` представляет ровно одну треть, а не приближение типа `Float64`.


## **30. Точная линейная система на Rational**


In [ ]:
A = [
    1//1  1//2
    1//3  1//1
]

b = [1//1, 1//1]

x = A \ b

@show x
@show A * x
@assert A * x == b


### Анализ

Результат является точным в рациональной арифметике — без ошибки округления, характерной для `Float64`.


## **31. Float64 vs Rational**


In [ ]:
A_float = Float64[
    1.0 0.5
    1/3 1.0
]

b_float = [1.0, 1.0]

x_float = A_float \ b_float

A_rat = [
    1//1 1//2
    1//3 1//1
]

b_rat = [1//1, 1//1]

x_rat = A_rat \ b_rat

@show x_float
@show x_rat


### Важно

Точность имеет свою цену.

Рациональная арифметика может быть значительно медленнее и приводить к росту числителей и знаменателей.

Выбор числового типа должен соответствовать цели вычислений.


## **32. Как выбрать факторизацию?**

| Задача | Предпочтительный инструмент |
|---|---|
| общая квадратная система | LU / `A \ b` |
| SPD-матрица | Cholesky |
| least squares | QR / `A \ b` |
| анализ ранга | SVD |
| низкоранговое сжатие | SVD |
| собственные значения симметричной матрицы | `eigen(Symmetric(A))` |
| общий анализ собственных значений | Schur / eigen |
| теория жордановых блоков | символьная / точная алгебра |


### Хорошая практика

Сначала используйте математическую структуру задачи, а уже затем выбирайте алгоритм.

Универсальный solver работает «более общо», но не всегда является лучшим выбором.


## **33. Типичные ошибки**

| Ошибка | Проблема | Правильный подход |
|---|---|---|
| `inv(A)*b` | ненужное вычисление обратной матрицы | `A\b` или факторизация |
| игнорирование pivoting в LU | неверное восстановление | учитывать перестановку |
| Cholesky для не-SPD матрицы | условия алгоритма не выполняются | `isposdef`, подходящая факторизация |
| normal equations в least squares | ухудшение обусловленности | QR / `A\b` |
| трактовка SVD как eigen | разные математические понятия | различать области применения |
| ожидание устойчивой Jordan form для Float64 | численная неустойчивость | Schur / symbolic |
| преобразование специальных структур в плотные | потеря информации и памяти | сохранять специальный тип |
| ожидание скорости Rational как у Float64 | exact arithmetic требует ресурсов | выбирать тип по задаче |


## **34. Практика**

### Задание 34.1 — собственные значения

Для матрицы:

```julia
A = [2.0 1.0; 1.0 2.0]
```

вычислите собственные значения.


In [ ]:
# Ваше решение:


### Пример решения 34.1


In [ ]:
A = [2.0 1.0; 1.0 2.0]

λ = eigvals(Symmetric(A))

@show λ


### Задание 34.2 — диагональная матрица собственных значений

Постройте `Diagonal(λ)`.


In [ ]:
# Ваше решение:


### Пример решения 34.2


In [ ]:
A = [2.0 1.0; 1.0 2.0]
λ = eigvals(Symmetric(A))
Λ = Diagonal(λ)

@show Λ


### Задание 34.3 — нижнетреугольная матрица

Создайте `LowerTriangular` из произвольной матрицы `3×3`.


In [ ]:
# Ваше решение:


### Пример решения 34.3


In [ ]:
A = reshape(1:9, 3, 3)
L = LowerTriangular(A)

@show L


### Задание 34.4 — выбор разложения

Выберите факторизацию для следующих случаев:

1. общая квадратная матрица;
2. SPD-матрица;
3. least squares;
4. низкоранговый анализ.


In [ ]:
# Odpowiedź tekstowa / komentarze:
#
# 1.
# 2.
# 3.
# 4.


### Пример решения 34.4

1. LU;
2. Cholesky;
3. QR;
4. SVD.


### Задание 34.5 — повторное использование факторизации

Для одной матрицы `A` решите три системы с разными правыми частями, вычислив факторизацию только один раз.


In [ ]:
# Ваше решение:


### Пример решения 34.5


In [ ]:
A = rand(5, 5)
F = lu(A)

B = rand(5, 3)
X = F \ B

@assert A * X ≈ B


## **35. Мини-проект — solver, выбирающий факторизацию**

### Цель

Создадим простую учебную функцию, которая использует информацию о структуре матрицы.


In [ ]:
function solve_structured(A::AbstractMatrix, b::AbstractVecOrMat)
    size(A, 1) == size(A, 2) || return qr(A) \ b

    if issymmetric(A) && isposdef(A)
        return cholesky(Symmetric(A)) \ b
    end

    return lu(A) \ b
end


### Тест 1 — SPD


In [ ]:
A1 = [
    4.0 1.0
    1.0 3.0
]
b1 = [1.0, 2.0]

x1 = solve_structured(A1, b1)

@assert A1 * x1 ≈ b1


### Тест 2 — общая матрица


In [ ]:
A2 = [
    1.0 2.0
    3.0 5.0
]
b2 = [1.0, 1.0]

x2 = solve_structured(A2, b2)

@assert A2 * x2 ≈ b2


### Тест 3 — least squares


In [ ]:
A3 = rand(6, 3)
b3 = rand(6)

x3 = solve_structured(A3, b3)

@show norm(A3 * x3 - b3)


### Анализ

Это не полноценный production-solver, но пример хорошо демонстрирует важный принцип:

> структура матрицы должна влиять на выбор алгоритма.


## **36. Итоговый checkpoint**

Ответьте без запуска кода:

1. Зачем используются факторизации?
2. Что означает `PA = LU`?
3. Почему LU использует pivoting?
4. Что даёт повторное использование факторизации?
5. Для каких задач особенно подходит QR?
6. Какие условия должна выполнять матрица для Cholesky?
7. Что возвращает SVD?
8. Для чего используется низкоранговая аппроксимация?
9. Что описывает EVD?
10. Почему `Symmetric(A)` полезна?
11. Что даёт Schur decomposition?
12. Почему Jordan form проблематична в численных вычислениях?
13. Зачем использовать `Diagonal` вместо полной матрицы?
14. Что даёт `SymTridiagonal`?
15. Что такое generic linear algebra?
16. Чем `1//3` отличается от `1/3`?
17. Почему Rational может давать точный результат?
18. Почему exact arithmetic требует больше ресурсов?
19. Какую факторизацию выбрать для SPD-матрицы?
20. Какую факторизацию выбрать для least squares?


## **37. Итоги урока**

Основные правила Lesson 12:

1. Факторизации являются фундаментом практической численной линейной алгебры.
2. LU — естественный выбор для общих квадратных систем.
3. Pivoting является частью устойчивой реализации LU.
4. Факторизацию следует повторно использовать для нескольких правых частей.
5. QR играет ключевую роль в least squares.
6. Cholesky очень эффективна для SPD-матриц.
7. SVD — один из наиболее универсальных инструментов анализа матриц.
8. EVD описывает собственные значения и собственные векторы квадратной матрицы.
9. Schur decomposition — важный устойчивый численный инструмент для задач на собственные значения.
10. Jordan form имеет прежде всего теоретическое и символьное значение, а не является стандартным численным методом для `Float64`.
11. Специальные типы матриц сохраняют математическую структуру и могут повышать производительность.
12. Julia поддерживает generic linear algebra для множества числовых типов.
13. Рациональные числа позволяют выполнять точные алгебраические вычисления.
14. Выбор факторизации должен определяться структурой задачи, а не привычкой.


## Источники для дальнейшего изучения

- Julia Standard Library — LinearAlgebra
- LinearAlgebra — Factorizations
- LinearAlgebra — Structured Matrices
- LinearAlgebra — Eigenvalues
- LinearAlgebra — Singular Value Decomposition
- LinearAlgebra — Schur Factorization
- Julia Manual — Mathematical Operations
- Julia Manual — Types


---

**Cartesian School · Julia Course**  
**Lesson 12 — Factorizations and Other Fun**  
**Автор:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School

[← Lesson 11 — Linear Algebra in Julia](Lesson_11_Linear_Algebra_in_Julia_Cartesian_School_RU.ipynb)  
[Оглавление](../README.ru.md)  
[Lesson 13 — Numerical Computing →](Lesson_13_Numerical_Computing_Julia_Cartesian_School_RU.ipynb)
